# Proyecto RappiPlus: de datos a decisiones de negocio

**Caso de estudio de análisis de datos.** Herramientas utilizadas: Python (pandas y statsmodels), SQL (PostgreSQL) y Power BI.

## Cómo consultar el notebook

Las tablas y conclusiones guardadas se pueden leer directamente. Esta copia incluye resultados agregados. Las celdas de exploración conservan el código que genera las vistas de registros.

Para volver a ejecutar los pasos 1 y 2 en Colab, carga los tres CSV de pedidos, catálogo y gasto de marketing con los nombres indicados abajo en `/content`. Los pasos 3 y 4 requieren acceso autorizado a PostgreSQL y una variable de entorno `RAPPIPLUS_DATABASE_URL` con una URL de SQLAlchemy para el controlador psycopg. El paso 5 carga el CSV del experimento desde la URL de origen indicada en el código.

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd

In [ ]:
# cargar archivos
orders = pd.read_csv("/content/rappiplus_orders_raw.csv")
catalog = pd.read_csv("/content/rappiplus_catalog.csv")
marketing = pd.read_csv("/content/rappiplus_marketing_spend.csv")

In [ ]:
# Inspeccionar registros, dimensiones, tipos y estadísticas iniciales de "Orders".
print(orders.head())
print("Filas y columnas:", orders.shape)
orders.info()
orders.describe()

In [ ]:
# Inspeccionar registros, dimensiones, tipos y estadísticas iniciales de "catalog".
print(catalog.head())
print("Filas y columnas:", catalog.shape)
catalog.info()
catalog.describe()

In [ ]:
# Inspeccionar registros, dimensiones, tipos y estadísticas iniciales de "Marketing".
print(marketing.head())
print("Filas y columnas:", marketing.shape)
marketing.info()
marketing.describe()

### 1.2. Observaciones de la exploración inicial

Se cargaron los datasets de pedidos (**25,100 filas y 12 columnas**),
catálogo (**7 filas y 4 columnas**) y marketing (**1,620 filas y 5 columnas**).

En pedidos se identificaron valores faltantes, cantidades negativas
y cantidades extremas que requieren revisión. En marketing se detectaron
**101 valores faltantes en `canal`**. El catálogo no presenta valores nulos.

Las fechas de pedidos y marketing están almacenadas como `object`;
su conversión y validación se realizarán durante la preparación de datos.

Estas observaciones orientan la revisión de calidad antes de calcular
ingresos, costos y rentabilidad. En esta etapa todavía no se han aplicado
correcciones ni se ha determinado el tratamiento de los valores detectados.

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
# Reiniciar las copias de trabajo desde los datos originales.
orders_work = orders.copy()
catalog_work = catalog.copy()
marketing_work = marketing.copy()

# Convertir las fechas. El catálogo no contiene columnas de fecha.
orders_work["fecha_hora_pedido"] = pd.to_datetime(
    orders_work["fecha_hora_pedido"], errors="coerce"
)

marketing_work["fecha"] = pd.to_datetime(
    marketing_work["fecha"], errors="coerce"
)

# Validar dimensiones, tipos y nulos.
print("Pedidos:", orders_work.shape)
print("Catálogo:", catalog_work.shape)
print("Marketing:", marketing_work.shape)

print("\nFecha de pedidos:", orders_work["fecha_hora_pedido"].dtype)
print("Nulos antes:", orders["fecha_hora_pedido"].isna().sum())
print("Nulos después:", orders_work["fecha_hora_pedido"].isna().sum())

print("\nFecha de marketing:", marketing_work["fecha"].dtype)
print("Nulos antes:", marketing["fecha"].isna().sum())
print("Nulos después:", marketing_work["fecha"].isna().sum())

Pedidos: (25100, 12)
Catálogo: (7, 4)
Marketing: (1620, 5)

Fecha de pedidos: datetime64[ns]
Nulos antes: 0
Nulos después: 0

Fecha de marketing: datetime64[ns]
Nulos antes: 0
Nulos después: 0


In [ ]:
# Diagnosticar las variables numéricas sin modificar los datos.
columnas_orders = [
    "cantidad", "precio_unitario", "monto_descuento", "monto_total"
]

for nombre, datos in [
    ("Pedidos", orders_work[columnas_orders]),
    ("Catálogo", catalog_work[["costo_unitario"]]),
    ("Marketing", marketing_work[["gasto"]])
]:
    print("\nDataset:", nombre)

    print("Nulos:")
    print(datos.isna().sum())

    print("Negativos:")
    print((datos < 0).sum())

    print("Ceros:")
    print((datos == 0).sum())


Dataset: Pedidos
Nulos:
cantidad           50
precio_unitario    50
monto_descuento    50
monto_total         0
dtype: int64
Negativos:
cantidad           4
precio_unitario    0
monto_descuento    0
monto_total        4
dtype: int64
Ceros:
cantidad               0
precio_unitario        0
monto_descuento    12551
monto_total            0
dtype: int64

Dataset: Catálogo
Nulos:
costo_unitario    0
dtype: int64
Negativos:
costo_unitario    0
dtype: int64
Ceros:
costo_unitario    0
dtype: int64

Dataset: Marketing
Nulos:
gasto    0
dtype: int64
Negativos:
gasto    0
dtype: int64
Ceros:
gasto    0
dtype: int64


In [ ]:
# Conservar los registros separados para documentar el tratamiento.
requiere_revision = (
    orders_work["cantidad"].isna()
    | orders_work["precio_unitario"].isna()
    | orders_work["monto_descuento"].isna()
    | (orders_work["cantidad"] < 0)
    | (orders_work["monto_total"] < 0)
)

orders_revision_numerica = orders_work[requiere_revision].copy()
filas_antes = len(orders_work)

# Excluir pedidos incompletos en las columnas críticas.
orders_work = orders_work.dropna(
    subset=["cantidad", "precio_unitario", "monto_descuento"]
).copy()

# Conservar compras con cantidad y monto positivos.
orders_work = orders_work[
    (orders_work["cantidad"] > 0)
    & (orders_work["monto_total"] > 0)
].copy()

# Validar el resultado.
print("Filas antes:", filas_antes)
print("Registros separados:", len(orders_revision_numerica))
print("Filas después:", len(orders_work))

print("\nNulos restantes:")
print(orders_work[columnas_orders].isna().sum())

print("\nNegativos restantes:")
print((orders_work[columnas_orders] < 0).sum())

print("\nCeros restantes:")
print((orders_work[columnas_orders] == 0).sum())

Filas antes: 25100
Registros separados: 54
Filas después: 25046

Nulos restantes:
cantidad           0
precio_unitario    0
monto_descuento    0
monto_total        0
dtype: int64

Negativos restantes:
cantidad           0
precio_unitario    0
monto_descuento    0
monto_total        0
dtype: int64

Ceros restantes:
cantidad               0
precio_unitario        0
monto_descuento    12550
monto_total            0
dtype: int64


In [ ]:
# Crear una tabla de comprobación sin modificar los pedidos.
revision_montos = orders_work[
    [
        "id_pedido", "cantidad", "precio_unitario",
        "monto_descuento", "monto_total"
    ]
].copy()

# Calcular el monto esperado según los componentes del pedido.
revision_montos["monto_esperado"] = (
    revision_montos["cantidad"] * revision_montos["precio_unitario"]
    - revision_montos["monto_descuento"]
).round(2)

# Comparar el monto registrado con el esperado.
revision_montos["diferencia"] = (
    revision_montos["monto_total"] - revision_montos["monto_esperado"]
).round(2)

print("Pedidos revisados:", len(revision_montos))
print("Coincidencias:", (revision_montos["diferencia"] == 0).sum())
print("Con diferencias:", (revision_montos["diferencia"] != 0).sum())

print("\nFrecuencia de las diferencias:")
print(revision_montos["diferencia"].value_counts())

# Mostrar ejemplos para revisar.
revision_montos[revision_montos["diferencia"] != 0].head(10)

Pedidos revisados: 25046
Coincidencias: 18790
Con diferencias: 6256

Frecuencia de las diferencias:
diferencia
 0.00    18790
-0.01     3170
 0.01     3086
Name: count, dtype: int64


In [ ]:
# Detectar e inspeccionar duplicados exactos en los tres datasets.
for nombre, datos in [
    ("Pedidos", orders_work),
    ("Catálogo", catalog_work),
    ("Marketing", marketing_work)
]:
    print("\nDataset:", nombre)
    print("Duplicados exactos:", datos.duplicated().sum())

    print("Ejemplos:")
    print(datos[datos.duplicated(keep=False)].head(6))

# Comprobar la unicidad de las claves según el contexto.
print("\nPedidos con id_pedido repetido:")
print(orders_work.duplicated(subset=["id_pedido"]).sum())

print("\nProductos repetidos en el catálogo:")
print(catalog_work.duplicated(subset=["nombre_producto"]).sum())

Dataset: Pedidos
Duplicados exactos: 100
Dataset: Catálogo
Duplicados exactos: 0
Dataset: Marketing
Duplicados exactos: 0
Pedidos con id_pedido repetido:
100
Productos repetidos en el catálogo:
0


In [ ]:
# Eliminar únicamente duplicados exactos de pedidos.
filas_antes = len(orders_work)

orders_work = (
    orders_work
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

# Validar la eliminación y la unicidad del pedido.
print("Filas antes:", filas_antes)
print("Filas eliminadas:", filas_antes - len(orders_work))
print("Filas después:", len(orders_work))
print("Duplicados exactos restantes:", orders_work.duplicated().sum())
print(
    "id_pedido repetidos:",
    orders_work.duplicated(subset=["id_pedido"]).sum()
)

Filas antes: 25046
Filas eliminadas: 100
Filas después: 24946
Duplicados exactos restantes: 0
id_pedido repetidos: 0


In [ ]:
# Inspeccionar categorías y nulos sin modificar los valores.
for nombre, datos, columnas in [
    (
        "Pedidos",
        orders_work,
        [
            "pais", "dispositivo", "fuente_referencia",
            "nombre_producto", "categoria_producto"
        ]
    ),
    (
        "Catálogo",
        catalog_work,
        ["nombre_producto", "categoria_producto", "proveedor"]
    ),
    (
        "Marketing",
        marketing_work,
        ["pais", "id_campaña", "canal"]
    )
]:
    print("\nDataset:", nombre)

    for columna in columnas:
        print("\nColumna:", columna)
        print(datos[columna].value_counts(dropna=False))

# Comprobar faltantes en los identificadores de pedidos y usuarios.
print("\nNulos en identificadores:")
print(orders_work[["id_pedido", "id_usuario"]].isna().sum())


Dataset: Pedidos

Columna: pais
pais
Colombia     7467
Mexico       7465
Argentina    7239
mexico        862
colombia      822
argentina     795
NaN           296
Name: count, dtype: int64

Columna: dispositivo
dispositivo
desktop    12681
mobile     12245
NaN           20
Name: count, dtype: int64

Columna: fuente_referencia
fuente_referencia
social         8385
organic        8269
paid_search    8262
NaN              30
Name: count, dtype: int64

Columna: nombre_producto
nombre_producto
Blender-XL-Red          4176
Vacuum-Pro-Black        4170
Jacket-Winter-M         4166
Sneakers-Urban-42       4129
Laptop-Gaming-16GB      2778
Tablet-Standard-64GB    2764
Phone-Pro-128GB         2733
NaN                       30
Name: count, dtype: int64

Columna: categoria_producto
categoria_producto
Hogar          8346
Moda           8295
Electronica    8275
NaN              30
Name: count, dtype: int64

Dataset: Catálogo

Columna: nombre_producto
nombre_producto
Laptop-Gaming-16GB      1
Phone-

In [ ]:
# Unificar la escritura de países en ambos datasets.
orders_work["pais"] = orders_work["pais"].str.strip().str.lower()
marketing_work["pais"] = marketing_work["pais"].str.strip().str.lower()

# Conservar los faltantes de contexto con una etiqueta explícita.
for columna in ["pais", "dispositivo", "fuente_referencia"]:
    orders_work[columna] = orders_work[columna].fillna("unknown")

marketing_work["canal"] = marketing_work["canal"].fillna("unknown")

# Validar las categorías modificadas.
print("Países en pedidos:")
print(orders_work["pais"].value_counts(dropna=False))

print("\nPaíses en marketing:")
print(marketing_work["pais"].value_counts(dropna=False))

print("\nNulos en las columnas tratadas de pedidos:")
print(
    orders_work[
        ["pais", "dispositivo", "fuente_referencia"]
    ].isna().sum()
)

print("\nCanales de marketing:")
print(marketing_work["canal"].value_counts(dropna=False))

# Mantener visible lo que aún falta resolver.
print("\nNulos pendientes en producto y categoría:")
print(
    orders_work[
        ["nombre_producto", "categoria_producto"]
    ].isna().sum()
)

Países en pedidos:
pais
mexico       8327
colombia     8289
argentina    8034
unknown       296
Name: count, dtype: int64

Países en marketing:
pais
mexico       540
colombia     540
argentina    540
Name: count, dtype: int64

Nulos en las columnas tratadas de pedidos:
pais                 0
dispositivo          0
fuente_referencia    0
dtype: int64

Canales de marketing:
canal
paid_search    507
organic        506
social         506
unknown        101
Name: count, dtype: int64

Nulos pendientes en producto y categoría:
nombre_producto       30
categoria_producto    30
dtype: int64


In [ ]:
# Conservar los pedidos cuyo producto no puede identificarse.
orders_revision_producto = orders_work[
    orders_work["nombre_producto"].isna()
].copy()

filas_antes = len(orders_work)

# Excluirlos de la base destinada al análisis conjunto de ingresos y costos.
orders_work = (
    orders_work
    .dropna(subset=["nombre_producto"])
    .reset_index(drop=True)
)

# Documentar la exclusión y verificar los faltantes restantes.
print("Filas antes:", filas_antes)
print("Pedidos separados:", len(orders_revision_producto))
print("Filas después:", len(orders_work))

print("\nMonto registrado en los pedidos separados:")
print(orders_revision_producto["monto_total"].sum())

print("\nNulos restantes en producto y categoría:")
print(
    orders_work[
        ["nombre_producto", "categoria_producto"]
    ].isna().sum()
)

Filas antes: 24946
Pedidos separados: 30
Filas después: 24916

Monto registrado en los pedidos separados:
12262.619999999999

Nulos restantes en producto y categoría:
nombre_producto       0
categoria_producto    0
dtype: int64


In [ ]:
# Validación final: no modifica los datasets.
for nombre, datos in [
    ("Pedidos", orders_work),
    ("Catálogo", catalog_work),
    ("Marketing", marketing_work)
]:
    print("\nDataset:", nombre)
    print("Filas y columnas:", datos.shape)
    print("Total de celdas nulas:", datos.isna().sum().sum())
    print("Duplicados exactos:", datos.duplicated().sum())

    print("Tipos de datos:")
    print(datos.dtypes)

print("\nid_pedido repetidos:")
print(orders_work.duplicated(subset=["id_pedido"]).sum())

print("\nProductos repetidos en catálogo:")
print(catalog_work.duplicated(subset=["nombre_producto"]).sum())


Dataset: Pedidos
Filas y columnas: (24916, 12)
Total de celdas nulas: 0
Duplicados exactos: 0
Tipos de datos:
id_pedido                     object
id_usuario                    object
fecha_hora_pedido     datetime64[ns]
pais                          object
dispositivo                   object
fuente_referencia             object
nombre_producto               object
categoria_producto            object
cantidad                     float64
precio_unitario              float64
monto_descuento              float64
monto_total                  float64
dtype: object

Dataset: Catálogo
Filas y columnas: (7, 4)
Total de celdas nulas: 0
Duplicados exactos: 0
Tipos de datos:
nombre_producto        object
categoria_producto     object
costo_unitario        float64
proveedor              object
dtype: object

Dataset: Marketing
Filas y columnas: (1620, 5)
Total de celdas nulas: 0
Duplicados exactos: 0
Tipos de datos:
fecha         datetime64[ns]
pais                  object
id_campaña           

### Conclusión de la revisión de calidad

Se validaron los tres datasets y se convirtieron las fechas a tipo
temporal. En pedidos se separaron 50 registros con datos numéricos
incompletos, cuatro con cantidades y montos negativos y 30 sin producto
identificable. También se eliminaron 100 duplicados exactos.

Se unificó la escritura de países y se identificaron como "unknown"
los faltantes de país, dispositivo, fuente de referencia y canal,
según correspondiera. Esta etiqueta conserva la ausencia de información.

La comprobación de montos no encontró diferencias superiores a un
centavo respecto de cantidad por precio menos descuento. Se conservaron
los importes pagados originales.

La base resultante contiene 24,916 pedidos, siete productos y 1,620
registros de marketing, sin celdas nulas ni duplicados exactos.

Los resultados posteriores corresponderán a los registros incluidos.
Las cantidades extremas identificadas se conservaron; su validez
operativa no está confirmada y deberá considerarse al interpretar
ingresos y rentabilidad.

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# Exportar los tres datasets limpios.
orders_work.to_csv("orders_clean.csv", index=False)
catalog_work.to_csv("catalog_clean.csv", index=False)
marketing_work.to_csv("marketing_clean.csv", index=False)

print("Exportación completada:")
print("orders_clean.csv:", orders_work.shape)
print("catalog_clean.csv:", catalog_work.shape)
print("marketing_clean.csv:", marketing_work.shape)

Exportación completada:
orders_clean.csv: (24916, 12)
catalog_clean.csv: (7, 4)
marketing_clean.csv: (1620, 5)


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# Calcular el ingreso total de los pedidos limpios.
revenue_total = orders_work["monto_total"].sum()

print(f"Ingreso total (revenue): {revenue_total:,.2f}")

Ingreso total (revenue): 51,954,718.94


In [ ]:
# Incorporar el costo unitario de cada producto.
orders_costos = orders_work.merge(
    catalog_work[["nombre_producto", "costo_unitario"]],
    on="nombre_producto",
    how="left",
    validate="many_to_one"
)

# Comprobar que todos los pedidos tengan un costo asignado.
assert orders_costos["costo_unitario"].notna().all(), \
    "Hay pedidos sin costo unitario en el catálogo."

# Calcular el costo de los productos vendidos.
orders_costos["costo_pedido"] = (
    orders_costos["cantidad"] * orders_costos["costo_unitario"]
)
costo_total = orders_costos["costo_pedido"].sum()

print("Pedidos incluidos:", len(orders_costos))
print(f"Costo total de productos vendidos: {costo_total:,.2f}")

Pedidos incluidos: 24916
Costo total de productos vendidos: 43,124,069.01


In [ ]:
# Calcular la inversión total en marketing.
marketing_total = marketing_work["gasto"].sum()

print(f"Inversión total en marketing: {marketing_total:,.2f}")

Inversión total en marketing: 2,871,843.53


In [ ]:
# Calcular el profit considerando productos y marketing.
profit_total = revenue_total - costo_total - marketing_total

print(f"Profit: {profit_total:,.2f}")

Profit: 5,958,806.40


In [ ]:
# Calcular el monto pagado promedio por pedido.
ticket_promedio = orders_work["monto_total"].mean()

print(f"Ticket promedio por orden: {ticket_promedio:,.2f}")

Ticket promedio por orden: 2,085.20


In [ ]:
# Calcular el promedio de unidades compradas por pedido.
cantidad_promedio = orders_work["cantidad"].mean()

print(f"Cantidad promedio de productos por orden: {cantidad_promedio:.2f}")

Cantidad promedio de productos por orden: 7.12


In [ ]:
# Comparar ingresos, costos y utilidad por producto.
rentabilidad_producto = (
    orders_costos.groupby("nombre_producto")
    .agg(
        ingresos=("monto_total", "sum"),
        costo_productos=("costo_pedido", "sum")
    )
)

rentabilidad_producto["utilidad_productos"] = (
    rentabilidad_producto["ingresos"]
    - rentabilidad_producto["costo_productos"]
)

rentabilidad_producto = rentabilidad_producto.sort_values(
    "utilidad_productos",
    ascending=False
)

print("Resultado por producto, antes de marketing:")
display(rentabilidad_producto.round(2))

Resultado por producto, antes de marketing:


,ingresos,costo_productos,utilidad_productos
nombre_producto,,,
Laptop-Gaming-16GB,43429372.98,40473494.64,2955878.34
Vacuum-Pro-Black,1621051.85,104314.40,1516737.45
Sneakers-Urban-42,1566075.93,106220.12,1459855.81
Phone-Pro-128GB,1057134.62,41896.80,1015237.82
Tablet-Standard-64GB,1062922.80,104697.13,958225.67
Blender-XL-Red,1611574.59,1109122.56,502452.03
Jacket-Winter-M,1606586.17,1184323.36,422262.81


In [ ]:
# Agrupar el gasto por canal y calcular su participación.
gasto_por_canal = (
    marketing_work.groupby("canal", as_index=False)["gasto"]
    .sum()
    .sort_values("gasto", ascending=False)
    .reset_index(drop=True)
)

gasto_por_canal["participacion_pct"] = (
    gasto_por_canal["gasto"] / gasto_por_canal["gasto"].sum() * 100
)

display(gasto_por_canal.round(2))

,canal,gasto,participacion_pct
0,social,918043.21,31.97
1,organic,913533.01,31.81
2,paid_search,863088.21,30.05
3,unknown,177179.10,6.17


### Gasto de marketing por canal

`social` registró el mayor gasto con 918,043.21 (31.97 % del total), seguido de `organic` con 913,533.01 (31.81 %) y `paid_search` con 863,088.21 (30.05 %). La diferencia entre `social` y `organic` fue de solo 0.16 puntos porcentuales.

El 6.17 % restante corresponde a registros cuyo canal no estaba identificado (`unknown`).

### Resumen de hallazgos

Por producto, los siete segmentos presentan utilidad positiva antes de
marketing. **Laptop-Gaming-16GB** aporta la mayor utilidad, con
**2,955,878.34**, seguido de **Vacuum-Pro-Black**, con **1,516,737.45**.
**Jacket-Winter-M** registra la menor utilidad, con **422,262.81**.
Esta comparación considera ingresos menos costo de productos vendidos;
el gasto de marketing se descuenta únicamente en el resultado global.

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
%pip install "psycopg[binary]"

import os

import pandas as pd
from sqlalchemy import create_engine

database_url = os.environ.get("RAPPIPLUS_DATABASE_URL")
if not database_url:
    raise RuntimeError(
        "Define RAPPIPLUS_DATABASE_URL con acceso autorizado a PostgreSQL."
    )

engine = create_engine(database_url, connect_args={"sslmode": "require"})


In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

In [ ]:
# Revisar la precisión de las fechas y la disponibilidad de sesiones.
query_orden_eventos = """
SELECT
    pg_typeof(timestamp_evento)::text AS tipo_fecha,
    COUNT(*) AS eventos,
    COUNT(*) FILTER (
        WHERE timestamp_evento IS NULL
    ) AS eventos_sin_fecha,
    COUNT(*) FILTER (
        WHERE timestamp_evento::timestamp::time <> TIME '00:00:00'
    ) AS eventos_fuera_de_medianoche,
    COUNT(*) FILTER (
        WHERE id_sesion IS NULL
    ) AS eventos_sin_sesion
FROM events
GROUP BY 1;
"""

pd.read_sql(query_orden_eventos, con=engine)

,tipo_fecha,eventos,eventos_sin_fecha,eventos_fuera_de_medianoche,eventos_sin_sesion
0,text,120000,0,0,0


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = """
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento;
"""

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,add_payment_info,6250
1,add_to_cart,7634
2,begin_checkout,7208
3,first_visit,7796
4,purchase,6240
5,select_item,7582


In [ ]:
# PARTE 1: Usuarios únicos por evento, ordenados según el flujo.
query_totals = """
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
    ELSE 7
END;
"""

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion ="""
WITH conteos AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
            ELSE 7
        END AS paso
    FROM events
    GROUP BY nombre_evento
),
comparacion AS (
    SELECT
        *,
        LAG(usuarios_unicos) OVER (
            ORDER BY paso
        ) AS usuarios_anteriores,
        FIRST_VALUE(usuarios_unicos) OVER (
            ORDER BY paso
        ) AS usuarios_iniciales
    FROM conteos
)
SELECT
    nombre_evento,
    usuarios_unicos,
    usuarios_anteriores - usuarios_unicos AS diferencia_usuarios,
    ROUND(
        100.0 * usuarios_unicos / NULLIF(usuarios_anteriores, 0),
        2
    ) AS conversion_paso_pct,
    ROUND(
        100.0 * usuarios_unicos / NULLIF(usuarios_iniciales, 0),
        2
    ) AS conversion_desde_inicio_pct
FROM comparacion
ORDER BY paso;
"""


conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,diferencia_usuarios,conversion_paso_pct,conversion_desde_inicio_pct
0,first_visit,7796,NaN,NaN,100.00
1,select_item,7582,214.0,97.26,97.26
2,add_to_cart,7634,-52.0,100.69,97.92
3,begin_checkout,7208,426.0,94.42,92.46
4,add_payment_info,6250,958.0,86.71,80.17
5,purchase,6240,10.0,99.84,80.04


In [ ]:
# Comparar recorridos con fechas estrictas y con empates de fecha.
query_embudo_secuencial = """
WITH RECURSIVE etapas(paso, evento) AS (
    VALUES
        (1, 'first_visit'),
        (2, 'select_item'),
        (3, 'add_to_cart'),
        (4, 'begin_checkout'),
        (5, 'add_payment_info'),
        (6, 'purchase')
),
fechas_por_usuario AS (
    SELECT
        e.id_usuario,
        p.paso,
        ARRAY_AGG(
            DISTINCT e.timestamp_evento::date
            ORDER BY e.timestamp_evento::date
        ) AS fechas
    FROM events AS e
    JOIN etapas AS p ON e.nombre_evento = p.evento
    WHERE e.id_usuario IS NOT NULL
      AND e.timestamp_evento IS NOT NULL
    GROUP BY e.id_usuario, p.paso
),
recorrido AS (
    SELECT
        f.id_usuario, f.paso, f.fechas[1] AS fecha,
        m.admite_empates
    FROM fechas_por_usuario AS f
    CROSS JOIN (VALUES (FALSE), (TRUE)) AS m(admite_empates)
    WHERE f.paso = 1

    UNION ALL

    SELECT r.id_usuario, f.paso, siguiente.fecha, r.admite_empates
    FROM recorrido AS r
    JOIN fechas_por_usuario AS f
      ON f.id_usuario = r.id_usuario AND f.paso = r.paso + 1
    CROSS JOIN LATERAL (
        SELECT MIN(d.fecha) AS fecha
        FROM UNNEST(f.fechas) AS d(fecha)
        WHERE d.fecha > r.fecha
           OR (r.admite_empates AND d.fecha = r.fecha)
    ) AS siguiente
    WHERE siguiente.fecha IS NOT NULL
)
SELECT
    p.paso,
    p.evento,
    COUNT(*) FILTER (
        WHERE r.admite_empates = FALSE
    ) AS usuarios_dias_distintos,
    COUNT(*) FILTER (
        WHERE r.admite_empates = TRUE
    ) AS usuarios_compatibles_por_fecha
FROM etapas AS p
LEFT JOIN recorrido AS r ON r.paso = p.paso
GROUP BY p.paso, p.evento
ORDER BY p.paso;
"""

embudo_secuencial = pd.read_sql(query_embudo_secuencial, con=engine)
display(embudo_secuencial)

,paso,evento,usuarios_dias_distintos,usuarios_compatibles_por_fecha
0,1,first_visit,7796,7796
1,2,select_item,6714,6728
2,3,add_to_cart,4935,4983
3,4,begin_checkout,2596,2663
4,5,add_payment_info,858,919
5,6,purchase,229,258


In [ ]:
# Calcular conversiones y caídas entre etapas.
conversion_embudo = embudo_secuencial.copy()

for columna, criterio in [
    ("usuarios_dias_distintos", "dias_distintos"),
    ("usuarios_compatibles_por_fecha", "fechas_compatibles"),
]:
    usuarios = conversion_embudo[columna]
    anterior = usuarios.shift()

    conversion_embudo[f"conversion_etapa_{criterio}_pct"] = (
        usuarios.div(anterior).mul(100).round(2)
    )
    conversion_embudo[f"caida_usuarios_{criterio}"] = (
        anterior - usuarios
    ).astype("Int64")
    conversion_embudo[f"conversion_desde_visita_{criterio}_pct"] = (
        usuarios.div(usuarios.iloc[0]).mul(100).round(2)
    )

display(conversion_embudo)

,paso,evento,usuarios_dias_distintos,usuarios_compatibles_por_fecha,conversion_etapa_dias_distintos_pct,caida_usuarios_dias_distintos,conversion_desde_visita_dias_distintos_pct,conversion_etapa_fechas_compatibles_pct,caida_usuarios_fechas_compatibles,conversion_desde_visita_fechas_compatibles_pct
0,1,first_visit,7796,7796,NaN,<NA>,100.00,NaN,<NA>,100.00
1,2,select_item,6714,6728,86.12,1082,86.12,86.30,1068,86.30
2,3,add_to_cart,4935,4983,73.50,1779,63.30,74.06,1745,63.92
3,4,begin_checkout,2596,2663,52.60,2339,33.30,53.44,2320,34.16
4,5,add_payment_info,858,919,33.05,1738,11.01,34.51,1744,11.79
5,6,purchase,229,258,26.69,629,2.94,28.07,661,3.31


---

### Resumen de hallazgos del funnel

El análisis inicial registró 7,796 usuarios únicos en primera visita y 6,240 en compra. El 80.04 % era la proporción entre esos conteos independientes por evento.

Al seguir a los mismos usuarios a través de las seis etapas, 229 de los 7,796 registraron la secuencia completa en fechas estrictamente crecientes (**2.94 %**). Al admitir eventos del mismo día, 258 usuarios tuvieron un recorrido compatible con las fechas registradas (**3.31 %**). El orden de los eventos dentro de un mismo día queda indeterminado.

Con el criterio de fechas estrictamente crecientes, la mayor caída en número de usuarios se observó entre **agregar al carrito e iniciar el checkout**: de 4,935 a 2,596 usuarios, una diferencia de **2,339**. La mayor caída porcentual se observó entre **agregar información de pago y comprar**: de 858 a 229 usuarios, una caída del **73.31 %** respecto a la etapa anterior. Ambos criterios de fecha identifican las mismas etapas críticas.

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = """
SELECT *
FROM user_activity;
"""
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final =  """
WITH usuarios_cohorte AS (
    SELECT
        id_usuario,
        CAST(
            DATE_TRUNC('month', CAST(fecha_registro AS DATE))
            AS DATE
        ) AS cohorte
    FROM users
),
retencion AS (
    SELECT
        u.cohorte,
        COUNT(DISTINCT u.id_usuario) AS clientes_iniciales,
        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 7 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w1,
        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 14 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w2,
        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 21 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w3
    FROM usuarios_cohorte AS u
    LEFT JOIN user_activity AS a
        ON u.id_usuario = a.id_usuario
    GROUP BY u.cohorte
)
SELECT
    cohorte,
    clientes_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(
        100.0 * retenido_w1 / NULLIF(clientes_iniciales, 0), 2
    ) AS semana_1,
    ROUND(
        100.0 * retenido_w2 / NULLIF(clientes_iniciales, 0), 2
    ) AS semana_2,
    ROUND(
        100.0 * retenido_w3 / NULLIF(clientes_iniciales, 0), 2
    ) AS semana_3
FROM retencion
ORDER BY cohorte;
"""


# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01,1687,695,676,706,41.20,40.07,41.85


In [ ]:
# Revisar el plazo de seguimiento disponible por cohorte.
query_ventana_cohortes = """
WITH corte AS (
    SELECT
        MAX(fecha_actividad::date) AS fecha_corte,
        MAX(dias_despues_registro) AS ultimo_dia_registrado
    FROM user_activity
)
SELECT
    DATE_TRUNC('month', u.fecha_registro::date)::date AS cohorte,
    COUNT(DISTINCT u.id_usuario) AS usuarios_registrados,
    MIN(c.fecha_corte - u.fecha_registro::date) AS dias_disponibles_minimos,
    c.fecha_corte,
    c.ultimo_dia_registrado
FROM users AS u
CROSS JOIN corte AS c
WHERE u.fecha_registro IS NOT NULL
GROUP BY 1, c.fecha_corte, c.ultimo_dia_registrado
ORDER BY 1;
"""

display(pd.read_sql(query_ventana_cohortes, con=engine))

,cohorte,usuarios_registrados,dias_disponibles_minimos,fecha_corte,ultimo_dia_registrado
0,2025-01-01,1627,148,2025-06-28,28
1,2025-02-01,1444,120,2025-06-28,28
2,2025-03-01,1636,89,2025-06-28,28
3,2025-04-01,1606,59,2025-06-28,28
4,2025-05-01,1687,28,2025-06-28,28


In [ ]:
# Calcular retención desde cada día de referencia hasta el día 28.
query_retencion_acumulada = """
WITH usuarios_cohorte AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', fecha_registro::date)::date AS cohorte
    FROM users
    WHERE fecha_registro IS NOT NULL
),
retencion AS (
    SELECT
        u.cohorte,
        COUNT(DISTINCT u.id_usuario) AS usuarios_registrados,
        COUNT(DISTINCT a.id_usuario) FILTER (
            WHERE a.dias_despues_registro BETWEEN 7 AND 28
        ) AS activos_d7_d28,
        COUNT(DISTINCT a.id_usuario) FILTER (
            WHERE a.dias_despues_registro BETWEEN 14 AND 28
        ) AS activos_d14_d28,
        COUNT(DISTINCT a.id_usuario) FILTER (
            WHERE a.dias_despues_registro BETWEEN 21 AND 28
        ) AS activos_d21_d28
    FROM usuarios_cohorte AS u
    LEFT JOIN user_activity AS a
        ON u.id_usuario = a.id_usuario
       AND a.activo = 1
       AND a.dias_despues_registro BETWEEN 7 AND 28
    GROUP BY u.cohorte
)
SELECT
    *,
    ROUND(100.0 * activos_d7_d28 / usuarios_registrados, 2) AS retencion_d7_d28_pct,
    ROUND(100.0 * activos_d14_d28 / usuarios_registrados, 2) AS retencion_d14_d28_pct,
    ROUND(100.0 * activos_d21_d28 / usuarios_registrados, 2) AS retencion_d21_d28_pct
FROM retencion
ORDER BY cohorte;
"""

retencion_acumulada = pd.read_sql(query_retencion_acumulada, con=engine)
display(retencion_acumulada)

,cohorte,usuarios_registrados,activos_d7_d28,activos_d14_d28,activos_d21_d28,retencion_d7_d28_pct,retencion_d14_d28_pct,retencion_d21_d28_pct
0,2025-01-01,1627,1381,1253,1027,84.88,77.01,63.12
1,2025-02-01,1444,1255,1154,940,86.91,79.92,65.10
2,2025-03-01,1636,1428,1306,1060,87.29,79.83,64.79
3,2025-04-01,1606,1394,1261,1022,86.80,78.52,63.64
4,2025-05-01,1687,1446,1321,1088,85.71,78.30,64.49


### Resumen de hallazgos de retención

**Actividad en días exactos.** En las cinco cohortes de enero a mayo de 2025, alrededor de 4 de cada 10 usuarios registraron actividad en los días de seguimiento. En el día 7, la proporción osciló entre **41.20 % y 42.84 %**. En el día 21, febrero registró el porcentaje más alto (**43.98 %**) y enero el más bajo (**40.32 %**).

**Retención con cierre común en el día 28.** Se contó una sola vez a cada usuario con al menos un registro activo dentro de la ventana. Entre cohortes, la proporción osciló entre **84.88 % y 87.29 %** para los días 7 a 28, entre **77.01 % y 79.92 %** para los días 14 a 28 y entre **63.12 % y 65.10 %** para los días 21 a 28.

En la ventana de los días 21 a 28, **febrero** mantuvo la mayor proporción de usuarios con actividad (**65.10 %**) y **enero** la menor (**63.12 %**). Esta medición complementa la del día 21 al incluir a quienes registraron actividad en cualquiera de los días hasta el cierre.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
  - **H₀ (hipótesis nula):** La tasa de conversión de compra es igual
  en control y tratamiento.
  - **H₁ (hipótesis alternativa):** La tasa de conversión de compra
  es diferente entre control y tratamiento.

**Test estadístico:** Prueba z bilateral para comparar dos
proporciones de grupos independientes.

**Nivel de significancia:** α = 0.05.

**Métrica principal:** `convirtio`, donde 1 indica compra completada
y 0 indica que no se completó.

In [ ]:
# tu código aquí
experiment = pd.read_csv("https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv")

display(experiment.head())
experiment.info()

print("\nUsuarios por variante:")
print(experiment["variante"].value_counts(dropna=False))

print("\nValores del indicador de conversión:")
print(experiment["convirtio"].value_counts(dropna=False))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB

Usuarios por variante:
variante
tratamiento    5035
control        4965
Name: count, dtype: int64

Valores del indicador de conversión:
convirtio
0    8401
1    1599
Name: count, dtype: int64


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# Verificar que cada usuario tenga una sola observación.
assert experiment["id_usuario"].is_unique, \
    "Hay usuarios repetidos; revisarlos antes de aplicar la prueba."

# Obtener conversiones y observaciones por variante.
resumen_ab = (
    experiment.groupby("variante")["convirtio"]
    .agg(usuarios="count", conversiones="sum")
    .reindex(["control", "tratamiento"])
)

resumen_ab["tasa_conversion_pct"] = (
    resumen_ab["conversiones"] / resumen_ab["usuarios"] * 100
)

display(resumen_ab.round(2))

# Aplicar la prueba z bilateral.
alpha = 0.05

z_stat, p_value = proportions_ztest(
    count=resumen_ab["conversiones"],
    nobs=resumen_ab["usuarios"],
    alternative="two-sided"
)

diferencia_pp = (
    resumen_ab.loc["tratamiento", "tasa_conversion_pct"]
    - resumen_ab.loc["control", "tasa_conversion_pct"]
)

print(f"Estadístico z: {z_stat:.4f}")
print(f"Valor p: {p_value:.6g}")
print(f"Diferencia tratamiento - control: {diferencia_pp:.2f} puntos porcentuales")

if p_value < alpha:
    print("Rechazamos H₀: hay evidencia de una diferencia en conversión.")
else:
    print("No rechazamos H₀: no hay evidencia suficiente de una diferencia.")

,usuarios,conversiones,tasa_conversion_pct
variante,,,
control,4965,779,15.69
tratamiento,5035,820,16.29


Estadístico z: -0.8133
Valor p: 0.416059
Diferencia tratamiento - control: 0.60 puntos porcentuales
No rechazamos H₀: no hay evidencia suficiente de una diferencia.


### Resumen de hallazgos del experimento A/B

El grupo control registró una conversión de **15.69 %**
(779 de 4,965 usuarios), mientras que tratamiento alcanzó **16.29 %**
(820 de 5,035 usuarios). La diferencia observada fue de
**0.60 puntos porcentuales a favor del tratamiento**.

La prueba z bilateral obtuvo un **valor p de 0.416059**, superior
al nivel de significancia de **0.05**. Por tanto, **no se rechaza H₀**:
no existe evidencia estadística suficiente para concluir que las
tasas de conversión sean diferentes. Esto no demuestra que sean iguales.

**Recomendación:** mantener el control como referencia. El experimento
no aporta evidencia suficiente para recomendar la nueva interfaz
por una mejora en conversión.

---

## Paso 6 · Comunicar el desempeño de ventas en Power BI

Construí el informe de ventas con los tres archivos generados en el paso 1: `orders_clean.csv`, `catalog_clean.csv` y `marketing_clean.csv`.

La vista ejecutiva reúne ingresos, utilidad de productos, ticket promedio, unidades por pedido, gasto de marketing y utilidad después de marketing. Incluye la evolución mensual y acumulada de los ingresos, una comparación de ingresos y utilidad por producto y un filtro de fechas.

El notebook conserva el análisis que alimenta el informe. El informe de Power BI se prepara como evidencia separada del notebook.


---